Fuel Efficiency Prediction — Clean Backend



In [3]:
!pip install lime --break-system-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 21.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=58f714e7af33ad1e42d3467f771fe0ba0738d051fa57651a6d7492a7f8694ef3
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [4]:
import sys
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
import sklearn
import shap
from lime.lime_tabular import LimeTabularExplainer
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold, StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error






In [5]:
RANDOM_STATE = 42

# CURRENT_YEAR is the reference year used to compute "vehicle/car age"
# features (car_age = CURRENT_YEAR - model_year / vehicle_age = CURRENT_YEAR - Year).
# It is fixed at 2026 because that is the year this analysis/revision was
# conducted. STATE THIS EXPLICITLY IN THE METHODS SECTION, e.g.:
#   "Vehicle age was computed as 2026 minus the model year, where 2026
#    denotes the reference year of this analysis."
# This single sentence pre-empts the "why 2026?" reviewer question.
CURRENT_YEAR = 2026

N_BOOTSTRAP  = 1000
rng          = np.random.default_rng(RANDOM_STATE)
kf           = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

XGB_PARAM_GRID = {
    "n_estimators":  [200, 300],
    "max_depth":     [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample":     [0.8, 1.0],
}

RF_PARAM_GRID = {
    "n_estimators":      [200, 300],
    "max_depth":         [8, 10, 12],
    "min_samples_split": [2, 5],
}


In [6]:

# ============================================================
# SHARED HELPER FUNCTIONS
# ============================================================

def report_metrics(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f"{name:26s} | R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
    return r2, rmse, mae


def bootstrap_r2_ci(y_true, y_pred, n_bootstrap=N_BOOTSTRAP):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    scores = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        scores[i] = r2_score(y_true[idx], y_pred[idx])
    lo, hi = np.percentile(scores, [2.5, 97.5])
    return lo, hi


def stratified_quantile_cv(model, X, y, n_splits=5, n_bins=5, random_state=RANDOM_STATE):
    """
    Plain KFold can produce highly variable folds when the target is
    heterogeneous (Car Specs spans economy hatchbacks to supercars, so a
    random fold can end up dominated by one segment). This bins the
    target into quantiles and uses StratifiedKFold on those bins, so
    every fold sees a similar mix of low/medium/high-MPG vehicles --
    giving a more reliable (lower-variance) CV estimate of generalization.
    """
    y_arr = np.array(y)
    X_arr = np.array(X)
    y_bins = pd.qcut(y_arr, q=n_bins, labels=False, duplicates="drop")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = []
    for train_idx, test_idx in skf.split(X_arr, y_bins):
        m = clone(model)
        m.fit(X_arr[train_idx], y_arr[train_idx])
        preds = m.predict(X_arr[test_idx])
        scores.append(r2_score(y_arr[test_idx], preds))
    return np.array(scores)


In [7]:

def tune_xgb(X_train_sc, y_train):
    grid = GridSearchCV(
        xgb.XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
        XGB_PARAM_GRID, cv=5, scoring="r2", n_jobs=-1
    )
    grid.fit(X_train_sc, y_train)
    return grid.best_estimator_, grid.best_params_


def tune_rf(X_train_sc, y_train):
    grid = GridSearchCV(
        RandomForestRegressor(random_state=RANDOM_STATE),
        RF_PARAM_GRID, cv=5, scoring="r2", n_jobs=-1
    )
    grid.fit(X_train_sc, y_train)
    return grid.best_estimator_, grid.best_params_


In [8]:

def plot_actual_vs_predicted(y_true, y_pred, title, filename):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]

    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.6, edgecolor="k")
    plt.plot(lims, lims, "r--", label="Ideal (y = x)")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()





In [9]:
def plot_residuals(y_true, y_pred, title_prefix, filename_scatter, filename_hist):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    residuals = y_true - y_pred

    plt.figure(figsize=(6, 5))
    plt.scatter(y_pred, residuals, alpha=0.6, edgecolor="k")
    plt.axhline(0, color="r", linestyle="--")
    plt.xlabel("Predicted")
    plt.ylabel("Residual (Actual - Predicted)")
    plt.title(f"{title_prefix} — Residual Scatter")
    plt.tight_layout()
    plt.savefig(filename_scatter, dpi=200)
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.hist(residuals, bins=25, edgecolor="k", alpha=0.7)
    plt.xlabel("Residual")
    plt.ylabel("Frequency")
    plt.title(f"{title_prefix} — Residual Distribution")
    plt.tight_layout()
    plt.savefig(filename_hist, dpi=200)
    plt.close()

    return residuals



In [10]:
def plot_tree_importance_comparison(rf_model, xgb_model, feature_names, title, filename):
    rf_imp  = rf_model.feature_importances_
    xgb_imp = xgb_model.feature_importances_
    rf_imp  = rf_imp / rf_imp.sum()
    xgb_imp = xgb_imp / xgb_imp.sum()

    x = np.arange(len(feature_names))
    width = 0.35
    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, rf_imp, width, label="Random Forest")
    plt.bar(x + width / 2, xgb_imp, width, label="XGBoost")
    plt.xticks(x, feature_names, rotation=45, ha="right")
    plt.ylabel("Normalized Feature Importance")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()





In [11]:
# ============================================================
# PART A: DATASET 1 — Auto MPG (Vehicle Configuration Model)
# ============================================================
print("=" * 60)
print("PART A: Auto MPG Dataset — Configuration-Based Model")
print("=" * 60)

df = pd.read_csv("auto-mpg.csv")
df["horsepower"] = df["horsepower"].replace("?", None).astype(float)
df = df.dropna()
df = df.drop(columns=["car name"], errors="ignore")

# Feature engineering (all derived from Auto MPG columns only)
df["power_to_weight"] = df["horsepower"] / df["weight"]
df["car_age"]         = CURRENT_YEAR - df["model-year"]
df["hp_per_cylinder"] = df["horsepower"] / df["cylinders"]

# FIX: target 'mpg' must NEVER appear in the feature list
#
# FIX (collinearity): 'model-year' is dropped. 'car_age' = CURRENT_YEAR -
# model-year is a perfect linear transform of it, so keeping both is
# redundant duplicate information -- this is exactly why 'car_age' showed
# ~0 SHAP/tree importance earlier (the model was splitting the same
# signal between the two). We keep 'car_age' (the engineered, more
# interpretable version) and drop the raw 'model-year' column.
config_features = [
    "cylinders", "displacement", "horsepower",
    "weight",
    "power_to_weight", "car_age", "hp_per_cylinder"
]

X_config = df[config_features]
y_config = df["mpg"]

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_config, y_config, test_size=0.2, random_state=RANDOM_STATE
)

scaler_config = StandardScaler()
X_c_train_sc  = scaler_config.fit_transform(X_c_train)
X_c_test_sc   = scaler_config.transform(X_c_test)

# --- Tune BOTH models ---
model_config, xgb_config_params = tune_xgb(X_c_train_sc, y_c_train)
rf_config, rf_config_params     = tune_rf(X_c_train_sc, y_c_train)
print("Best Params (Config XGBoost):", xgb_config_params)
print("Best Params (Config Random Forest):", rf_config_params)

# --- 5-Fold CV for both ---
cv_r2_xgb_c = cross_val_score(model_config, X_c_train_sc, y_c_train, cv=kf, scoring="r2")
cv_r2_rf_c  = cross_val_score(rf_config,    X_c_train_sc, y_c_train, cv=kf, scoring="r2")
print(f"Config XGBoost 5-Fold CV R2:      {cv_r2_xgb_c.mean():.4f} +/- {cv_r2_xgb_c.std():.4f}")
print(f"Config RandomForest 5-Fold CV R2: {cv_r2_rf_c.mean():.4f} +/- {cv_r2_rf_c.std():.4f}")

# --- Test metrics for both ---
preds_c_xgb = model_config.predict(X_c_test_sc)
preds_c_rf  = rf_config.predict(X_c_test_sc)
r2_c_xgb, rmse_c_xgb, mae_c_xgb = report_metrics("Config XGBoost (test)", y_c_test, preds_c_xgb)
r2_c_rf,  rmse_c_rf,  mae_c_rf  = report_metrics("Config RandomForest (test)", y_c_test, preds_c_rf)

# --- Bootstrap CI (primary model = XGBoost) ---
ci_lo_c, ci_hi_c = bootstrap_r2_ci(y_c_test, preds_c_xgb)
print(f"Config XGBoost Bootstrap 95% CI for R2: [{ci_lo_c:.4f}, {ci_hi_c:.4f}]")

# --- Actual vs Predicted + Residual plots (primary model) ---
plot_actual_vs_predicted(
    y_c_test, preds_c_xgb,
    "Actual vs Predicted — Config Model (XGBoost)",
    "actual_vs_predicted_config.png"
)
plot_residuals(
    y_c_test, preds_c_xgb, "Config Model (XGBoost)",
    "residual_scatter_config.png", "residual_distribution_config.png"
)

# --- RF vs XGBoost native tree-importance comparison ---
plot_tree_importance_comparison(
    rf_config, model_config, config_features,
    "Tree Feature Importance: RF vs XGBoost — Config Model",
    "tree_importance_comparison_config.png"
)

joblib.dump(model_config,    "model_config.pkl")
joblib.dump(rf_config,       "model_config_rf.pkl")
joblib.dump(scaler_config,   "scaler_config.pkl")
joblib.dump(config_features, "features_config.pkl")
print("Model 1 (Config, XGBoost primary + RF benchmark) saved.\n")


PART A: Auto MPG Dataset — Configuration-Based Model
Best Params (Config XGBoost): {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
Best Params (Config Random Forest): {'max_depth': 12, 'min_samples_split': 5, 'n_estimators': 300}
Config XGBoost 5-Fold CV R2:      0.8573 +/- 0.0362
Config RandomForest 5-Fold CV R2: 0.8629 +/- 0.0264
Config XGBoost (test)      | R2: 0.8400 | RMSE: 2.8809 | MAE: 1.9802
Config RandomForest (test) | R2: 0.8640 | RMSE: 2.6561 | MAE: 1.8107
Config XGBoost Bootstrap 95% CI for R2: [0.7296, 0.9149]
Model 1 (Config, XGBoost primary + RF benchmark) saved.



In [12]:

# ============================================================
# PART B: DATASET 2 — Car Specifications (Brand + Model)
# ============================================================
print("=" * 60)
print("PART B: Car Specifications Dataset — Brand-Based Model")
print("=" * 60)

from google.colab import files
uploaded = files.upload()   # select archive.zip

import zipfile
with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall("car_dataset")

df2 = pd.read_csv("car_dataset/data.csv")
df2.columns = df2.columns.str.strip()

df2 = df2.dropna(subset=["highway MPG", "city mpg", "Engine HP", "Engine Cylinders", "Year"])
df2 = df2[df2["Engine HP"] > 0]
df2 = df2[df2["Engine Cylinders"] > 0]

# avg_mpg is computed ONLY here, from Car-Specs highway/city MPG.
# This is Model 2's target — it is NOT derived from, and never mixed
# with, the Auto MPG dataset used in Part A.
df2["avg_mpg"] = (df2["highway MPG"] * 0.55) + (df2["city mpg"] * 0.45)


def estimate_weight(hp, cyl):
    base = 2500 + (hp - 100) * 8 + (cyl - 4) * 200
    return max(1800, min(6000, base))


df2["weight_est"]       = df2.apply(lambda r: estimate_weight(r["Engine HP"], r["Engine Cylinders"]), axis=1)
df2["vehicle_age"]      = CURRENT_YEAR - df2["Year"]
df2["power_to_weight"]  = df2["Engine HP"] / df2["weight_est"]
df2["hp_per_cylinder"]  = df2["Engine HP"] / df2["Engine Cylinders"]
df2["displacement_est"] = df2["Engine Cylinders"] * 400

df2 = df2.replace([np.inf, -np.inf], np.nan)
df2 = df2.dropna(subset=[
    "Engine HP", "Engine Cylinders", "weight_est", "displacement_est",
    "vehicle_age", "power_to_weight", "hp_per_cylinder", "avg_mpg"
])

# FIX: MSRP removed from the predictive feature set entirely.
# Make / Model / MSRP stay in df2 for dashboard lookup only (see
# predict_by_brand below) — they are never passed into the model.
#
# FIX (collinearity): 'displacement_est' is dropped from the feature
# list. It was computed as Engine Cylinders * 400 -- a fixed multiple of
# a feature already in the model, so it carries zero additional
# information. This is exactly why it showed ~0 SHAP/tree importance
# above. displacement_est is still computed on df2 below (kept as a
# descriptive/derived column in case it's referenced elsewhere), it is
# simply excluded from the model's input features.
brand_features = [
    "Engine HP", "Engine Cylinders", "weight_est",
    "vehicle_age",
    "power_to_weight", "hp_per_cylinder"
]

X_brand = df2[brand_features].copy()
y_brand = df2["avg_mpg"]

X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_brand, y_brand, test_size=0.2, random_state=RANDOM_STATE
)

scaler_brand = StandardScaler()
X_b_train_sc = scaler_brand.fit_transform(X_b_train)
X_b_test_sc  = scaler_brand.transform(X_b_test)

# --- Tune BOTH sub-models ---
xgb_brand, xgb_brand_params = tune_xgb(X_b_train_sc, y_b_train)
rf_brand,  rf_brand_params  = tune_rf(X_b_train_sc, y_b_train)
print("Best Params (Brand XGBoost):", xgb_brand_params)
print("Best Params (Brand Random Forest):", rf_brand_params)

# --- 5-Fold CV for both sub-models ---
cv_r2_xgb = cross_val_score(xgb_brand, X_b_train_sc, y_b_train, cv=kf, scoring="r2")
cv_r2_rf  = cross_val_score(rf_brand,  X_b_train_sc, y_b_train, cv=kf, scoring="r2")
print(f"Brand XGBoost 5-Fold CV R2:      {cv_r2_xgb.mean():.4f} +/- {cv_r2_xgb.std():.4f}")
print(f"Brand RandomForest 5-Fold CV R2: {cv_r2_rf.mean():.4f} +/- {cv_r2_rf.std():.4f}")

# --- Stratified (quantile-binned) CV: more reliable estimate for a
# heterogeneous target like avg_mpg (economy cars vs. supercars can land
# unevenly across plain random folds, inflating the std of plain KFold) ---
cv_r2_xgb_strat = stratified_quantile_cv(xgb_brand, X_b_train_sc, y_b_train)
cv_r2_rf_strat  = stratified_quantile_cv(rf_brand,  X_b_train_sc, y_b_train)
print(f"Brand XGBoost Stratified CV R2:      {cv_r2_xgb_strat.mean():.4f} +/- {cv_r2_xgb_strat.std():.4f}")
print(f"Brand RandomForest Stratified CV R2: {cv_r2_rf_strat.mean():.4f} +/- {cv_r2_rf_strat.std():.4f}")

# --- Weighted ensemble (as in original design) ---
xgb_b_preds = xgb_brand.predict(X_b_test_sc)
rf_b_preds  = rf_brand.predict(X_b_test_sc)

XGB_W, RF_W = 0.60, 0.40
ens_preds = (XGB_W * xgb_b_preds) + (RF_W * rf_b_preds)

xgb_r2, xgb_rmse, xgb_mae = report_metrics("Brand XGBoost", y_b_test, xgb_b_preds)
rf_r2,  rf_rmse,  rf_mae  = report_metrics("Brand RandomForest", y_b_test, rf_b_preds)
ens_r2, ens_rmse, ens_mae = report_metrics("Brand Ensemble (60/40)", y_b_test, ens_preds)

ci_lo_b, ci_hi_b = bootstrap_r2_ci(y_b_test, ens_preds)
print(f"Brand Ensemble Bootstrap 95% CI for R2: [{ci_lo_b:.4f}, {ci_hi_b:.4f}]")

# --- Actual vs Predicted + Residual plots (final ensemble) ---
plot_actual_vs_predicted(
    y_b_test, ens_preds,
    "Actual vs Predicted — Brand Model (Ensemble)",
    "actual_vs_predicted_brand.png"
)
plot_residuals(
    y_b_test, ens_preds, "Brand Model (Ensemble)",
    "residual_scatter_brand.png", "residual_distribution_brand.png"
)

# --- RF vs XGBoost native tree-importance comparison ---
plot_tree_importance_comparison(
    rf_brand, xgb_brand, brand_features,
    "Tree Feature Importance: RF vs XGBoost — Brand Model",
    "tree_importance_comparison_brand.png"
)

joblib.dump(xgb_brand,      "model_xgb_brand.pkl")
joblib.dump(rf_brand,       "model_rf_brand.pkl")
joblib.dump(scaler_brand,   "scaler_brand.pkl")
joblib.dump(brand_features, "features_brand.pkl")

brand_metrics = {
    "xgb_r2": round(xgb_r2, 4), "xgb_rmse": round(xgb_rmse, 4), "xgb_mae": round(xgb_mae, 4),
    "rf_r2":  round(rf_r2, 4),  "rf_rmse":  round(rf_rmse, 4),  "rf_mae":  round(rf_mae, 4),
    "ens_r2": round(ens_r2, 4), "ens_rmse": round(ens_rmse, 4), "ens_mae": round(ens_mae, 4),
    "xgb_weight": XGB_W, "rf_weight": RF_W,
}
joblib.dump(brand_metrics, "brand_metrics.pkl")
print("Model 2 (Brand: XGBoost + RF + Ensemble) saved.\n")

PART B: Car Specifications Dataset — Brand-Based Model


Saving archive.zip to archive.zip
Best Params (Brand XGBoost): {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 1.0}
Best Params (Brand Random Forest): {'max_depth': 12, 'min_samples_split': 2, 'n_estimators': 200}
Brand XGBoost 5-Fold CV R2:      0.8232 +/- 0.1118
Brand RandomForest 5-Fold CV R2: 0.8202 +/- 0.1119
Brand XGBoost Stratified CV R2:      0.8230 +/- 0.1071
Brand RandomForest Stratified CV R2: 0.8159 +/- 0.1059
Brand XGBoost              | R2: 0.8848 | RMSE: 2.0207 | MAE: 1.2862
Brand RandomForest         | R2: 0.8850 | RMSE: 2.0196 | MAE: 1.2710
Brand Ensemble (60/40)     | R2: 0.8876 | RMSE: 1.9964 | MAE: 1.2707
Brand Ensemble Bootstrap 95% CI for R2: [0.8691, 0.9038]
Model 2 (Brand: XGBoost + RF + Ensemble) saved.



In [13]:

# ============================================================
# PART C: SHAP EXPLAINABILITY
# (Paper 1 reports ONLY this section)
# ============================================================
print("=" * 60)
print("PART C: SHAP Explainability")
print("=" * 60)

explainer_config   = shap.TreeExplainer(model_config)
X_c_test_df        = pd.DataFrame(X_c_test_sc, columns=config_features)
shap_values_config  = explainer_config(X_c_test_df)

plt.figure()
shap.summary_plot(shap_values_config, X_c_test_df, show=False)
plt.title("SHAP Summary — Config Model (Auto MPG)")
plt.tight_layout()
plt.savefig("shap_summary_config.png", dpi=200)
plt.close()

plt.figure()
shap.summary_plot(shap_values_config, X_c_test_df, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Bar) — Config Model")
plt.tight_layout()
plt.savefig("shap_bar_config.png", dpi=200)
plt.close()

plt.figure()
shap.plots.waterfall(shap_values_config[0], show=False)
plt.tight_layout()
plt.savefig("shap_local_config_instance0.png", dpi=200)
plt.close()

# Explain the XGBoost sub-model (primary contributor, 60% ensemble weight)
explainer_brand    = shap.TreeExplainer(xgb_brand)
X_b_test_df        = pd.DataFrame(X_b_test_sc, columns=brand_features)
shap_values_brand   = explainer_brand(X_b_test_df)

plt.figure()
shap.summary_plot(shap_values_brand, X_b_test_df, show=False)
plt.title("SHAP Summary — Brand Model (Car Specifications)")
plt.tight_layout()
plt.savefig("shap_summary_brand.png", dpi=200)
plt.close()

plt.figure()
shap.summary_plot(shap_values_brand, X_b_test_df, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Bar) — Brand Model")
plt.tight_layout()
plt.savefig("shap_bar_brand.png", dpi=200)
plt.close()

plt.figure()
shap.plots.waterfall(shap_values_brand[0], show=False)
plt.tight_layout()
plt.savefig("shap_local_brand_instance0.png", dpi=200)
plt.close()

print("SHAP plots saved: shap_summary_config.png, shap_bar_config.png, "
      "shap_local_config_instance0.png, shap_summary_brand.png, "
      "shap_bar_brand.png, shap_local_brand_instance0.png\n")


PART C: SHAP Explainability
SHAP plots saved: shap_summary_config.png, shap_bar_config.png, shap_local_config_instance0.png, shap_summary_brand.png, shap_bar_brand.png, shap_local_brand_instance0.png



In [ ]:


# ============================================================
# PART C2: LIME EXPLAINABILITY
# (Project backend / Paper 2 only — Paper 1 does not report this)
# ============================================================
print("=" * 60)
print("PART C2: LIME Explainability")
print("=" * 60)

lime_explainer_config = LimeTabularExplainer(
    training_data=X_c_train_sc, feature_names=config_features,
    mode="regression", discretize_continuous=True, random_state=RANDOM_STATE,
)
lime_exp_config_0 = lime_explainer_config.explain_instance(
    X_c_test_sc[0], model_config.predict, num_features=len(config_features)
)
lime_exp_config_0.save_to_file("lime_local_config_instance0.html")
fig = lime_exp_config_0.as_pyplot_figure()
fig.suptitle("LIME Local Explanation — Config Model (instance 0)")
fig.tight_layout()
fig.savefig("lime_local_config_instance0.png", dpi=200)
plt.close(fig)

# Explaining the XGBoost sub-model (same model SHAP explained above) so
# the SHAP vs LIME comparison in Part C3 is apples-to-apples.
lime_explainer_brand = LimeTabularExplainer(
    training_data=X_b_train_sc, feature_names=brand_features,
    mode="regression", discretize_continuous=True, random_state=RANDOM_STATE,
)
lime_exp_brand_0 = lime_explainer_brand.explain_instance(
    X_b_test_sc[0], xgb_brand.predict, num_features=len(brand_features)
)
lime_exp_brand_0.save_to_file("lime_local_brand_instance0.html")
fig = lime_exp_brand_0.as_pyplot_figure()
fig.suptitle("LIME Local Explanation — Brand Model (instance 0)")
fig.tight_layout()
fig.savefig("lime_local_brand_instance0.png", dpi=200)
plt.close(fig)

print("LIME local explanations saved (html + png) for both models.\n")


def lime_global_importance(explainer, predict_fn, X_test_sc, feature_names, n_samples=50):
    """Aggregate |LIME weight| across instances -> approximate global importance."""
    n = min(n_samples, len(X_test_sc))
    importance_sum = {f: 0.0 for f in feature_names}
    for i in range(n):
        exp = explainer.explain_instance(X_test_sc[i], predict_fn, num_features=len(feature_names))
        for feat_desc, weight in exp.as_list():
            for f in feature_names:
                if f in feat_desc:
                    importance_sum[f] += abs(weight)
                    break
    return {f: v / n for f, v in importance_sum.items()}


print("Aggregating LIME global importance over 50 test instances (Config Model)...")
lime_importance_config = lime_global_importance(
    lime_explainer_config, model_config.predict, X_c_test_sc, config_features, n_samples=50
)
print("Aggregating LIME global importance over 50 test instances (Brand Model)...")
lime_importance_brand = lime_global_importance(
    lime_explainer_brand, xgb_brand.predict, X_b_test_sc, brand_features, n_samples=50
)

PART C2: LIME Explainability
LIME local explanations saved (html + png) for both models.

Aggregating LIME global importance over 50 test instances (Config Model)...
Aggregating LIME global importance over 50 test instances (Brand Model)...


In [ ]:


# ============================================================
# PART C3: SHAP vs LIME COMPARISON (Paper 2's core contribution)
# ============================================================
print("=" * 60)
print("PART C3: SHAP vs LIME Comparison")
print("=" * 60)


def compare_shap_lime(shap_values, feature_names, lime_importance_dict, title, filename):
    shap_importance = np.abs(shap_values.values).mean(axis=0)
    shap_dict = dict(zip(feature_names, shap_importance))

    shap_total = sum(shap_dict.values())
    lime_total = sum(lime_importance_dict.values())
    shap_norm = {f: v / shap_total for f, v in shap_dict.items()}
    lime_norm = {f: v / lime_total for f, v in lime_importance_dict.items()}

    rho, pval = spearmanr(
        [shap_norm[f] for f in feature_names],
        [lime_norm[f] for f in feature_names],
    )
    print(f"{title} | Spearman rank correlation (SHAP vs LIME): rho={rho:.4f}, p={pval:.4f}")

    x = np.arange(len(feature_names))
    width = 0.35
    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, [shap_norm[f] for f in feature_names], width, label="SHAP (normalized)")
    plt.bar(x + width / 2, [lime_norm[f] for f in feature_names], width, label="LIME (normalized)")
    plt.xticks(x, feature_names, rotation=45, ha="right")
    plt.ylabel("Normalized Importance")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()

    return rho, pval


rho_config, p_config = compare_shap_lime(
    shap_values_config, config_features, lime_importance_config,
    "SHAP vs LIME — Config Model (Auto MPG)", "shap_lime_comparison_config.png"
)
rho_brand, p_brand = compare_shap_lime(
    shap_values_brand, brand_features, lime_importance_brand,
    "SHAP vs LIME — Brand Model (Car Specifications)", "shap_lime_comparison_brand.png"
)

xai_comparison_results = {
    "config_spearman_rho": round(float(rho_config), 4),
    "config_spearman_p":   round(float(p_config), 4),
    "brand_spearman_rho":  round(float(rho_brand), 4),
    "brand_spearman_p":    round(float(p_brand), 4),
}
joblib.dump(xai_comparison_results, "xai_comparison_results.pkl")
print("SHAP vs LIME comparison saved.\n")




PART C3: SHAP vs LIME Comparison
SHAP vs LIME — Config Model (Auto MPG) | Spearman rank correlation (SHAP vs LIME): rho=0.8571, p=0.0137
SHAP vs LIME — Brand Model (Car Specifications) | Spearman rank correlation (SHAP vs LIME): rho=0.8857, p=0.0188
SHAP vs LIME comparison saved.



In [ ]:

# ============================================================
# PART D: PREDICTION FUNCTIONS
# ============================================================

def predict_by_config(cylinders, displacement, horsepower, weight, model_year):
    power_to_weight = horsepower / weight
    car_age         = CURRENT_YEAR - model_year
    hp_per_cylinder = horsepower / cylinders

    input_data = pd.DataFrame([[
        cylinders, displacement, horsepower,
        weight,
        power_to_weight, car_age, hp_per_cylinder
    ]], columns=config_features)

    input_scaled = scaler_config.transform(input_data)
    mpg_pred     = model_config.predict(input_scaled)[0]
    kmpl_pred    = mpg_pred * 0.4251

    print(f"\n{'=' * 35}\n [Config-Based Prediction]\n MPG : {mpg_pred:.2f}\n KMPL: {kmpl_pred:.2f}\n{'=' * 35}")
    return mpg_pred, kmpl_pred


def predict_by_brand(brand_name, model_name):
    match = df2[
        (df2["Make"].str.lower()  == brand_name.lower()) &
        (df2["Model"].str.lower() == model_name.lower())
    ]
    if match.empty:
        print(f"Car not found: {brand_name} {model_name}")
        return None

    row = match.iloc[0]
    input_data   = pd.DataFrame([[row[c] for c in brand_features]], columns=brand_features)
    input_scaled = scaler_brand.transform(input_data)

    xgb_pred = xgb_brand.predict(input_scaled)[0]
    rf_pred  = rf_brand.predict(input_scaled)[0]
    ens_pred = (XGB_W * xgb_pred) + (RF_W * rf_pred)
    kmpl_pred = ens_pred * 0.4251

    # MSRP shown here ONLY as dashboard lookup metadata — never a model input
    msrp = row["MSRP"] if "MSRP" in df2.columns else None

    msg = (f"\n{'=' * 35}\n [Brand-Based Prediction]\n Car : {brand_name} {model_name}"
           f"\n MPG : {ens_pred:.2f}\n KMPL: {kmpl_pred:.2f}")
    if msrp is not None:
        msg += f"\n MSRP (lookup only, not a model feature): {msrp}"
    msg += f"\n{'=' * 35}"
    print(msg)
    return ens_pred, kmpl_pred


predict_by_config(cylinders=4, displacement=1500, horsepower=120, weight=2800, model_year=2020)
predict_by_brand("BMW", "1 Series")





 [Config-Based Prediction]
 MPG : 28.02
 KMPL: 11.91

 [Brand-Based Prediction]
 Car : BMW 1 Series
 MPG : 21.35
 KMPL: 9.07
 MSRP (lookup only, not a model feature): 40650


(np.float64(21.346522429791577), np.float64(9.0744066849044))

In [ ]:

# ============================================================
# PART E: REPRODUCIBILITY INFO
# ============================================================
print("=" * 60)
print("PART E: Reproducibility Information")
print("=" * 60)

try:
    import lime as lime_pkg
    lime_version = getattr(lime_pkg, "__version__", "unknown")
except Exception:
    lime_version = "unknown"

repro_info = {
    "python_version":   sys.version.split()[0],
    "platform":         platform.platform(),
    "processor":        platform.processor() or "unknown (typical Colab runtime: Intel Xeon CPU)",
    "sklearn_version":  sklearn.__version__,
    "xgboost_version":  xgb.__version__,
    "shap_version":     shap.__version__,
    "lime_version":     lime_version,
    "numpy_version":    np.__version__,
    "pandas_version":   pd.__version__,
    "random_state":     RANDOM_STATE,
    "current_year_used_for_age_features": CURRENT_YEAR,
    "environment_note": "Google Colab (standard CPU runtime, RAM ~12-13GB unless upgraded)",
}
for k, v in repro_info.items():
    print(f"{k}: {v}")

joblib.dump(repro_info, "reproducibility_info.pkl")
with open("reproducibility_info.txt", "w") as f:
    for k, v in repro_info.items():
        f.write(f"{k}: {v}\n")

print("\nReproducibility info saved: reproducibility_info.txt / .pkl\n")


PART E: Reproducibility Information
python_version: 3.12.13
platform: Linux-6.6.122+-x86_64-with-glibc2.35
processor: x86_64
sklearn_version: 1.6.1
xgboost_version: 3.3.0
shap_version: 0.52.0
lime_version: unknown
numpy_version: 2.0.2
pandas_version: 2.2.2
random_state: 42
current_year_used_for_age_features: 2026
environment_note: Google Colab (standard CPU runtime, RAM ~12-13GB unless upgraded)

Reproducibility info saved: reproducibility_info.txt / .pkl



In [ ]:
# ============================================================
# PART F: DOWNLOAD ARTIFACTS
# ============================================================
for f in [
    "model_config.pkl", "model_config_rf.pkl", "scaler_config.pkl", "features_config.pkl",
    "model_xgb_brand.pkl", "model_rf_brand.pkl", "scaler_brand.pkl", "features_brand.pkl",
    "brand_metrics.pkl",
    "actual_vs_predicted_config.png", "residual_scatter_config.png", "residual_distribution_config.png",
    "actual_vs_predicted_brand.png", "residual_scatter_brand.png", "residual_distribution_brand.png",
    "tree_importance_comparison_config.png", "tree_importance_comparison_brand.png",
    "shap_summary_config.png", "shap_bar_config.png", "shap_local_config_instance0.png",
    "shap_summary_brand.png", "shap_bar_brand.png", "shap_local_brand_instance0.png",
    "lime_local_config_instance0.png", "lime_local_config_instance0.html",
    "lime_local_brand_instance0.png", "lime_local_brand_instance0.html",
    "shap_lime_comparison_config.png", "shap_lime_comparison_brand.png",
    "xai_comparison_results.pkl",
    "reproducibility_info.txt", "reproducibility_info.pkl",
]:
    files.download(f)

print("All artifacts downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All artifacts downloaded!


In [14]:
for f in [
 "shap_bar_brand.png",
]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>